# 21. Erreichbarkeit (freiwillig)

Kann die Schildkröte Kassiopeia jedes weiße Feld ihres Landes erreichen? Wie viele Inseln hat eine Karte? Steht fest, welcher
Container der schwerste ist? Alle drei Fragen haben denselben Kern: **Was kann man von einem Startpunkt aus erreichen**, wenn
man nur bestimmte Schritte machen darf?

Die Antwort liefert ein Verfahren, das du vielleicht vom Malprogramm kennst: Der **Farbeimer** füllt von einem Punkt aus alles,
was zusammenhängt. Auf Englisch heißt das **Flood Fill** (Überflutung). In diesem Notebook lernst du:

- wie Flood Fill mit einer Liste als **Stapel** funktioniert,
- wie du damit prüfst, ob alles erreichbar ist,
- wie du zusammenhängende **Gruppen** zählst,
- dass das nicht nur auf Karten funktioniert, sondern für alle „wer erreicht wen“-Fragen.

Dieses Notebook ist freiwillig. Es setzt die Notebooks 12 (Raster) und 15 (Mengen) voraus.

## 1. Flood Fill mit einem Stapel

Die Idee: Du hast einen **Stapel** mit Feldern, die du noch untersuchen musst, und eine Menge `besucht` mit allen Feldern, die du
schon gefunden hast. Am Anfang liegt nur das Startfeld auf dem Stapel.

1. Nimm das oberste Feld vom Stapel.
2. Sieh dir seine Nachbarn an. Jeden Nachbarn, der frei ist und noch nicht besucht wurde, markierst du als besucht und legst
   ihn auf den Stapel.
3. Wiederhole das, bis der Stapel leer ist.

Dann enthält `besucht` alle erreichbaren Felder. Kein Feld wird doppelt untersucht, weil es nur beim ersten Finden auf den
Stapel kommt.

In Python ist der Stapel eine Liste: `stapel.append(feld)` legt ein Feld oben drauf, `stapel.pop()` nimmt das oberste
herunter und gibt es zurück. Ein Feld speicherst du als **Tupel** `(z, s)` (Notebook 20). Listen dürfen nicht in eine Menge,
Tupel schon, weil sie sich nicht verändern können.

In [ ]:
karte = ['#########',
         '#  #    #',
         '#  # #  #',
         '#  K #  #',
         '#    #  #',
         '#########']
hoehe = len(karte)
breite = len(karte[0])
dz = [-1, 1, 0, 0]
ds = [0, 0, -1, 1]


def erreichbar(karte, start):
    besucht = {start}
    stapel = [start]
    while len(stapel) > 0:
        z, s = stapel.pop()
        for r in range(4):
            z2 = z + dz[r]
            s2 = s + ds[r]
            if 0 <= z2 < len(karte) and 0 <= s2 < len(karte[0]) and karte[z2][s2] != '#' and (z2, s2) not in besucht:
                besucht.add((z2, s2))
                stapel.append((z2, s2))
    return besucht


felder = erreichbar(karte, (3, 3))
print(len(felder), 'Felder erreichbar')
for z in range(hoehe):
    zeile = ''
    for s in range(breite):
        if (z, s) in felder:
            zeile = zeile + 'o'
        else:
            zeile = zeile + karte[z][s]
    print(zeile)

#### Frage 1.1

Was würde passieren, wenn man ein Feld erst beim **Herunternehmen** vom Stapel als besucht markiert und nicht schon beim
Drauflegen? Und was, wenn man `besucht` ganz weglässt?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Markiert man erst beim Herunternehmen, kann dasselbe Feld mehrmals auf den Stapel kommen, nämlich von jedem seiner Nachbarn
aus, bevor es untersucht wird. Das Ergebnis stimmt trotzdem, aber es wird mehr gerechnet. Ohne `besucht` dagegen läuft man
ewig hin und her: von A nach B, von B wieder nach A und so weiter. Das ist die Endlosschleife aus Notebook 13, und `besucht`
ist wieder die Menge „schon gesehen“.

</details>

## 2. Ist alles erreichbar?

Damit kannst du die Frage der Aufgabe „Kassiopeia“ beantworten: Zähle alle weißen Felder und vergleiche mit der Anzahl der
erreichbaren. Sind es gleich viele, kann Kassiopeia alle erreichen.

#### Aufgabe 2.1

Schreibe `alles_erreichbar(karte)`: Finde Kassiopeias Feld `K`, zähle die weißen Felder (alles außer `#`) und vergleiche mit
`erreichbar`. Teste mit der Karte aus Abschnitt 1 und mit `karte2`, in der eine Wand dazugekommen ist.

**Erwartete Ausgabe:**
```
True False
```

In [ ]:
karte2 = ['#########',
          '#  # #  #',
          '#  # #  #',
          '#  K #  #',
          '#    #  #',
          '#########']


def alles_erreichbar(karte):
    return False        # ersetze diese Zeile


print(alles_erreichbar(karte), alles_erreichbar(karte2))

<details>
<summary>Tipp anzeigen</summary>

Zwei Schleifen über alle Felder: weiße Felder zählen und dabei `start = (z, s)` für das `K` merken. Dann
`len(erreichbar(karte, start)) == weisse`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
karte2 = ['#########',
          '#  # #  #',
          '#  # #  #',
          '#  K #  #',
          '#    #  #',
          '#########']


def alles_erreichbar(karte):
    weisse = 0
    start = None
    for z in range(len(karte)):
        for s in range(len(karte[0])):
            if karte[z][s] != '#':
                weisse = weisse + 1
            if karte[z][s] == 'K':
                start = (z, s)
    return len(erreichbar(karte, start)) == weisse


print(alles_erreichbar(karte), alles_erreichbar(karte2))
```

In `karte2` versperrt das neue `#` in der zweiten Zeile den einzigen Durchgang nach rechts. Die beiden Spalten ganz rechts
kann Kassiopeia nicht mehr erreichen.

</details>

## 3. Gruppen zählen

Wie viele **Inseln** hat eine Karte? Eine Insel ist eine Gruppe von Landfeldern `#`, die über Nachbarn (Norden, Süden, Westen,
Osten) zusammenhängen. Die Idee: Geh alle Felder durch. Findest du ein Landfeld, das noch zu keiner Insel gehört, hast du eine
**neue** Insel gefunden. Dann markierst du mit Flood Fill die ganze Insel als besucht, damit du sie nicht noch einmal zählst.

Dafür brauchst du ein Flood Fill, das nicht über freie Felder, sondern über **Landfelder** läuft. Am einfachsten gibst du der
Funktion als Parameter mit, welches Zeichen begehbar ist.

#### Aufgabe 3.1

Schreibe `gebiet(karte, start, zeichen)`, die alle Felder findet, die vom Start aus über Felder mit `zeichen` erreichbar sind
(wie `erreichbar`, nur mit `karte[z2][s2] == zeichen`). Zähle damit die Inseln und gib ihre Größen aus.

**Erwartete Ausgabe:**
```
6 Inseln mit den Größen [3, 1, 1, 1, 2, 2]
```

In [ ]:
inseln = ['##..#',
          '#..#.',
          '..#..',
          '.....',
          '##.##']


def gebiet(karte, start, zeichen):
    return {start}      # ersetze diese Zeile


schon_gezaehlt = set()
groessen = []
# dein Code

<details>
<summary>Tipp anzeigen</summary>

`gebiet` ist `erreichbar` mit der Bedingung `karte[z2][s2] == zeichen`. Für das Zählen: Bei jedem Landfeld, das noch nicht in
`schon_gezaehlt` ist, `insel = gebiet(inseln, (z, s), '#')`, die Größe merken und alle Felder der Insel zu `schon_gezaehlt`
hinzufügen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
inseln = ['##..#',
          '#..#.',
          '..#..',
          '.....',
          '##.##']


def gebiet(karte, start, zeichen):
    besucht = {start}
    stapel = [start]
    while len(stapel) > 0:
        z, s = stapel.pop()
        for r in range(4):
            z2 = z + dz[r]
            s2 = s + ds[r]
            if 0 <= z2 < len(karte) and 0 <= s2 < len(karte[0]) and karte[z2][s2] == zeichen and (z2, s2) not in besucht:
                besucht.add((z2, s2))
                stapel.append((z2, s2))
    return besucht


schon_gezaehlt = set()
groessen = []
for z in range(len(inseln)):
    for s in range(len(inseln[0])):
        if inseln[z][s] == '#' and (z, s) not in schon_gezaehlt:
            insel = gebiet(inseln, (z, s), '#')
            groessen.append(len(insel))
            for feld in insel:
                schon_gezaehlt.add(feld)
print(len(groessen), 'Inseln mit den Größen', groessen)
```

Die drei einzelnen Landfelder oben rechts und in der Mitte berühren sich nur **schräg**. Weil schräge Nachbarn hier nicht
zählen, sind es drei eigene Inseln. Ob schräg zählt, steht in der Aufgabe (oder muss entschieden werden, Notebook 23). Mit acht
Nachbarn (Notebook 12) wären die drei eine einzige Insel, und es gäbe nur 4 Inseln.

</details>

## 4. Erreichbarkeit ohne Karte

Flood Fill braucht keine Karte. Es genügt, für jedes Ding zu wissen, welche Dinge man von ihm aus **direkt** erreicht. Das
speicherst du in einem Wörterbuch (Notebook 15), zum Beispiel: Von Stadt A fährt ein Bus nach B und C. So ein Netz aus Dingen
und Verbindungen heißt **Graph**. Die „Nachbarn“ eines Dings sind dann einfach `verbindungen.get(ding, [])`.

In [ ]:
verbindungen = {'A': ['B', 'C'], 'B': ['D'], 'C': ['D'], 'D': [], 'E': ['A']}


def erreichbar_graph(verbindungen, start):
    besucht = {start}
    stapel = [start]
    while len(stapel) > 0:
        ding = stapel.pop()
        for nachbar in verbindungen.get(ding, []):
            if nachbar not in besucht:
                besucht.add(nachbar)
                stapel.append(nachbar)
    return besucht


print(sorted(erreichbar_graph(verbindungen, 'A')))
print(sorted(erreichbar_graph(verbindungen, 'E')))

## Wettbewerbs-Ecke 1: Kassiopeia

*Angelehnt an die Junioraufgabe „Kassiopeia“ (Bundeswettbewerb Informatik 2015).*

Jetzt die echten Dateien: `daten/kassiopeia.txt` ist das Beispiel vom Aufgabenblatt (richtige Antwort „Ja“), `daten/kassiopeia1.txt`
ein weiteres Beispiel der Aufgabe. In der ersten Zeile stehen die Anzahl der Zeilen und der Spalten.

#### Aufgabe E.1

Lies beide Dateien ein (mit Auffüllen zu kurzer Zeilen, Notebook 12) und gib für jede `Ja` oder `Nein` aus.

**Erwartete Ausgabe:**
```
daten/kassiopeia.txt: Ja
    #########
    #  #    #
    #  # #  #
    #  K #  #
    #    #  #
    #########
daten/kassiopeia1.txt: Nein
    ###########
    #     #   #
    #   ###   #
    #   #     #
    # K ###   #
    #     #   #
    ###########
```

In [ ]:
def lies_welt(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zeilen = f.read().splitlines()
    teile = zeilen[0].split()
    hoehe = int(teile[0])
    breite = int(teile[1])
    welt = []
    for z in range(1, hoehe + 1):
        welt.append(zeilen[z] + ' ' * (breite - len(zeilen[z])))
    return welt


# dein Code

<details>
<summary>Tipp anzeigen</summary>

Für jede Datei `welt = lies_welt(name)` und `alles_erreichbar(welt)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lies_welt(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zeilen = f.read().splitlines()
    teile = zeilen[0].split()
    hoehe = int(teile[0])
    breite = int(teile[1])
    welt = []
    for z in range(1, hoehe + 1):
        welt.append(zeilen[z] + ' ' * (breite - len(zeilen[z])))
    return welt


for name in ['daten/kassiopeia.txt', 'daten/kassiopeia1.txt']:
    welt = lies_welt(name)
    if alles_erreichbar(welt):
        print(name + ': Ja')
    else:
        print(name + ': Nein')
    for zeile in welt:
        print('   ', zeile)
```

In `kassiopeia1.txt` trennt eine Wand das Land in zwei Teile. Ein Bild mit den erreichbaren Feldern (Notebook 19) wäre für die
Dokumentation die überzeugendste Begründung. Die Aufgabe hat noch eine Fortsetzung: Kann Kassiopeia alle Felder so besuchen,
dass sie keines zweimal betritt? Das ist viel schwerer. Siehe `jwinf3/34_Kassiopeia/kassiopeia_loesung.ipynb`.

</details>

## Wettbewerbs-Ecke 2: Container, die Gegenprobe

*Angelehnt an die Junioraufgabe „Container“ (Jugendwettbewerb 2022).*

In Notebook 15 hast du die Kandidaten für den schwersten Container gefunden: alle, die nie leichter waren. Jetzt die
**Gegenprobe**: Ein Container ist **sicher** der schwerste, wenn man von ihm aus über „ist schwerer als“ **alle anderen**
erreicht. Ist A schwerer als B und B schwerer als C, dann ist A auch schwerer als C, obwohl A und C nie zusammen gewogen wurden.

#### Aufgabe E.2

Baue für jede Datei ein Wörterbuch `schwerer_als`, das jedem Container die Liste der Container zuordnet, die nachweislich leichter
sind (direkt gewogen). Prüfe dann mit `erreichbar_graph` für jeden Container, ob er alle anderen erreicht, und gib aus, welcher
sicher der schwerste ist.

**Erwartete Ausgabe:**
```
daten/container0.txt: sicher der schwerste: []
daten/container1.txt: sicher der schwerste: [4]
```

In [ ]:
def lies_paare(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        paare = []
        for zeile in f.read().splitlines():
            teile = zeile.split()
            if len(teile) == 2:
                paare.append([int(teile[0]), int(teile[1])])
    return paare


# dein Code

<details>
<summary>Tipp anzeigen</summary>

Gruppier-Muster aus Notebook 15: `schwerer_als[a].append(b)`. Ein Container erreicht alle, wenn `erreichbar_graph` so viele
Container liefert, wie es insgesamt gibt (er selbst ist ja mitgezählt).

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lies_paare(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        paare = []
        for zeile in f.read().splitlines():
            teile = zeile.split()
            if len(teile) == 2:
                paare.append([int(teile[0]), int(teile[1])])
    return paare


for name in ['daten/container0.txt', 'daten/container1.txt']:
    schwerer_als = {}
    alle = set()
    for a, b in lies_paare(name):
        if a not in schwerer_als:
            schwerer_als[a] = []
        schwerer_als[a].append(b)
        alle.add(a)
        alle.add(b)
    sicher = []
    for c in sorted(alle):
        if len(erreichbar_graph(schwerer_als, c)) == len(alle):
            sicher.append(c)
    print(name + ': sicher der schwerste:', sicher)
```

Für `container1.txt` erreicht Container 4 alle anderen, er ist also sicher der schwerste. Für `container0.txt` erreicht keiner
alle: Es steht nicht fest. Das ist dasselbe Ergebnis wie in Notebook 15, aber diesmal **direkt bewiesen**. Zwei Wege mit
gleichem Ergebnis sind eine gute Kontrolle (Notebook 14).

</details>

## Für Neugierige: Kürzeste Wege

Nimmt man statt des obersten das **älteste** Feld (`stapel.pop(0)`), wird aus dem Stapel eine **Warteschlange**. Dann breitet sich
die Suche wie eine Welle gleichmäßig vom Start aus. Das heißt **Breitensuche**. Merkt man sich dabei für jedes Feld, in welcher
Welle es gefunden wurde, erhält man die Länge des **kürzesten Weges** vom Start zu jedem Feld. Damit lassen sich Labyrinthe
lösen.

## Das hast du gelernt

| Werkzeug | So geht es |
|---|---|
| Stapel | Liste mit `append` (drauflegen) und `pop()` (oberstes herunternehmen) |
| Flood Fill | Start in `besucht` und auf den Stapel; solange der Stapel nicht leer ist: herunternehmen, neue Nachbarn markieren und drauflegen |
| alles erreichbar? | Anzahl der erreichbaren mit der Anzahl aller vergleichen |
| Gruppen zählen | jedes noch nicht gezählte Feld startet eine neue Gruppe, Flood Fill markiert die ganze Gruppe |
| Graph | Wörterbuch „Ding → Liste der direkt erreichbaren Dinge“ |

- Felder als Tupel `(z, s)`, damit sie in eine Menge dürfen.
- Ohne `besucht` läuft die Suche endlos im Kreis.